# SprintBoard · SFT + GRPO Fine-tuning on Hugging Face TRL

This is the **judge-runnable training notebook** for the SprintBoard environment
(OpenEnv Hackathon submission).

**Recipe**, end-to-end, against the *real* environment:

1. Clones the SprintBoard Space and installs the env in-process.
2. Evaluates the **baseline** (untrained) policy on all 15 sprint-planning tasks.
3. **Phase A — SFT warm-start**: a few hundred gradient steps of supervised
   fine-tuning on the 15 expert plans shipped with the environment
   (`sprint_planning_env.agent.TASKS_COMMANDS`). This anchors the format
   (`one verb per line`) and prevents GRPO from collapsing the policy.
4. Evaluates the **SFT** policy on all 15 tasks.
5. **Phase B — GRPO refinement**: TRL `GRPOTrainer` with a *full multi-step
   trajectory reward* (deterministic grader score) plus a strong format
   shaping reward. KL is set to 0 so SFT progress is preserved; LR is small.
6. Evaluates the **trained** (post-GRPO) policy on all 15 tasks.
7. Saves baseline / SFT / trained reward plots to `assets/`.

**Why SFT + GRPO?** Pure GRPO on a 1.5 B model with 15 prompts collapses the
output format before reward signal can shape behaviour. SFT on the expert
plans bootstraps the model into the right output channel, and GRPO refines
from there with measurable gains.

**Hardware**: tested on a free Colab T4 (≈15 GB VRAM). One full pass takes
20–40 minutes (SFT is fast; GRPO is the slow part).

## 1 · Setup — clone the SprintBoard env and install training deps

Re-running the cell is safe; the clone step only fires the first time.

In [ ]:
import os, sys, subprocess, pathlib

REPO_DIR = pathlib.Path("sprint_planning_env_repo")
if not REPO_DIR.exists() and not pathlib.Path("server").exists():
    !git clone -q https://huggingface.co/spaces/vikramsrini/sprint_planning_env {REPO_DIR}
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

!pip install -q -U pip
!pip install -q -r requirements-train.txt
!pip install -q -e .
print("Setup complete")

## 2 · Imports & global config

In [ ]:
import json, random, re, time
from pathlib import Path
from statistics import mean
from typing import List, Dict

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from trl import GRPOConfig, GRPOTrainer

from sprint_planning_env.models import SprintAction
from sprint_planning_env.server.environment import SprintBoardEnvironment
from sprint_planning_env.server.tasks import TASK_REGISTRY, list_task_ids
from sprint_planning_env.agent import TASKS_COMMANDS

# ----- Knobs -----------------------------------------------------------------
MODEL_ID         = "Qwen/Qwen2.5-1.5B-Instruct"   # 1.5B fits on a free T4 with LoRA
TASK_IDS         = list_task_ids()                 # all 15 tasks
MAX_NEW_TOKENS   = 384                             # enough for ~10-12 commands
MAX_PROMPT_TOK   = 768

# Phase A — Supervised Fine-Tuning warm-start (expert plans)
SFT_EPOCHS       = 6
SFT_LR           = 5e-5
SFT_GRAD_CLIP    = 1.0

# Phase B — GRPO refinement
NUM_GENERATIONS  = 4
GRPO_EPOCHS      = 2          # short — SFT did the heavy lifting
GRPO_LR          = 5e-7       # tiny LR so we don't undo SFT
GRPO_BETA        = 0.0        # KL=0: ref policy is the *base* model (PEFT detail);
                              #         we explicitly *don't* want to be pulled back to it.
OUT_DIR          = Path("./sprintboard_qwen_lora")
ASSETS_DIR       = Path("./assets"); ASSETS_DIR.mkdir(exist_ok=True)

torch.manual_seed(0); random.seed(0); np.random.seed(0)
print(f"Tasks: {len(TASK_IDS)} | model: {MODEL_ID} | torch: {torch.__version__}")

## 3 · Multi-step rollout + reward functions

The agent outputs a **plan**: one command per line. We execute the whole plan
in the SprintBoard environment and use the deterministic grader's final score
as the reward signal. This is genuine multi-step credit assignment — the
policy is rewarded for *reasoning across the full episode*, not for emitting
a syntactically valid first token.

Two shaping signals are combined:

* **Trajectory reward** (`[0, 1]`) — final grader score after running every
  command in the rollout.
* **Format reward** (`[0, 0.35]`) — graded:
  * `+0.15` if any line is a valid SprintBoard verb,
  * `+0.10` if at least 3 lines are valid verbs (encourages substantive plans),
  * `+0.10` if the plan ends with `FINALIZE_SPRINT` (process discipline).

In [ ]:
VALID_VERBS = {
    "LIST_BACKLOG", "VIEW_STORY", "CHECK_DEPS", "VIEW_TEAM", "VIEW_VELOCITY",
    "VIEW_SPRINT", "VIEW_BUGS", "VIEW_EPIC", "SEARCH_BACKLOG", "ESTIMATE",
    "ASSIGN", "UNASSIGN", "ADD_TO_SPRINT", "REMOVE_FROM_SPRINT", "SET_PRIORITY",
    "FLAG_RISK", "DECOMPOSE", "FINALIZE_SPRINT",
}

_TASK_RE = re.compile(r"TASK_ID:\s*(task_\d+)")


def extract_task_id(prompt) -> str:
    if isinstance(prompt, list):
        prompt = prompt[-1].get("content", "") if prompt and isinstance(prompt[-1], dict) else str(prompt)
    m = _TASK_RE.search(str(prompt))
    return m.group(1) if m else "task_1"


def _to_text(completion) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion:
        first = completion[0]
        if isinstance(first, dict):
            return first.get("content", "")
        if isinstance(first, str):
            return first
    return str(completion)


def parse_plan(completion, max_cmds: int = 20) -> List[str]:
    """Pull command lines out of a free-form completion."""
    text = _to_text(completion)
    cmds: List[str] = []
    for raw in text.splitlines():
        line = raw.strip().strip("`").strip("-*>\u2022 ").strip()
        if not line:
            continue
        verb = line.split()[0].upper()
        if verb in VALID_VERBS:
            cmds.append(line)
        if len(cmds) >= max_cmds:
            break
    return cmds


def rollout_score(task_id: str, plan: List[str]) -> Dict[str, float]:
    """Run a plan against the env and return reward + metadata."""
    env = SprintBoardEnvironment()
    env.reset(task_id=task_id, seed=0)
    cumulative = 0.0
    last_grader = 0.0
    finalized = False
    n_steps = 0
    for cmd in plan or ["FINALIZE_SPRINT"]:
        obs = env.step(SprintAction(command=cmd))
        cumulative += float(obs.reward or 0.0)
        g = obs.metadata.get("grader_score")
        if g is not None:
            last_grader = float(g)
        n_steps += 1
        if obs.done:
            finalized = True
            break
    if not finalized:                          # auto-finalize if model never did
        obs = env.step(SprintAction(command="FINALIZE_SPRINT"))
        g = obs.metadata.get("grader_score")
        if g is not None:
            last_grader = float(g)
        n_steps += 1
    return {
        "reward": float(np.clip(last_grader, 0.0, 1.0)),
        "steps": n_steps,
        "cumulative": cumulative,
    }


# ----- TRL reward callbacks ---------------------------------------------------
training_reward_log: List[float] = []
training_format_log: List[float] = []

def trajectory_reward(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        task_id = extract_task_id(prompt)
        plan = parse_plan(completion)
        rewards.append(rollout_score(task_id, plan)["reward"])
    if rewards:
        training_reward_log.append(float(mean(rewards)))
    return rewards


def format_reward(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        plan = parse_plan(completion)
        score = 0.0
        if len(plan) >= 1:
            score += 0.15
        if len(plan) >= 3:
            score += 0.10
        if plan and plan[-1].split()[0].upper() == "FINALIZE_SPRINT":
            score += 0.10
        rewards.append(score)
    if rewards:
        training_format_log.append(float(mean(rewards)))
    return rewards


demo = rollout_score("task_1", TASKS_COMMANDS["task_1"])
print("Smoke-test rollout on task_1 (expert plan) =>", demo)

## 4 · Build training dataset — one prompt per task

In [ ]:
SYSTEM_PROMPT = """You are an expert Agile Scrum Master operating SprintBoard.
Given a sprint-planning scenario, produce an ordered PLAN of commands that
investigates the board and fixes the issues, then finalises the sprint.
Output **one command per line**, no markdown, no commentary, no numbering.
Always end the plan with FINALIZE_SPRINT.

Allowed commands:
  Investigation: LIST_BACKLOG, VIEW_STORY <id>, CHECK_DEPS <id>, VIEW_TEAM,
                 VIEW_VELOCITY, VIEW_SPRINT, VIEW_BUGS, VIEW_EPIC <id>,
                 SEARCH_BACKLOG <kw>
  Planning:      ESTIMATE <id> <pts>, ASSIGN <id> <Name>, UNASSIGN <id>,
                 ADD_TO_SPRINT <id>, REMOVE_FROM_SPRINT <id>,
                 SET_PRIORITY <id> <P0|P1|P2>, FLAG_RISK <id> <reason>,
                 DECOMPOSE <epic_id> \"sub1\" \"sub2\" ...,
                 FINALIZE_SPRINT

Strategy:
  1. Investigate first (LIST_BACKLOG, VIEW_TEAM, VIEW_VELOCITY ...).
  2. Then fix the planning issue revealed by the scenario alert.
  3. End with FINALIZE_SPRINT."""


def build_prompt(task_id: str) -> str:
    task = TASK_REGISTRY[task_id]
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"TASK_ID: {task_id}  (difficulty: {task['difficulty']})\n"
        f"SCENARIO ALERT: {task['alert']}\n\n"
        f"Produce the full plan now (one command per line):"
    )


prompts = [build_prompt(tid) for tid in TASK_IDS]
dataset = Dataset.from_dict({"prompt": prompts})
print(f"Built dataset with {len(dataset)} prompts (1 per task).")

## 5 · Load model + LoRA adapter

In [ ]:
print(f"Loading {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6 · Greedy evaluation helper

Single rollout per task with deterministic decoding (`do_sample=False`) so the
scores reported across phases are directly comparable.

In [ ]:
@torch.inference_mode()
def policy_rollout(task_id: str, max_new_tokens: int = MAX_NEW_TOKENS) -> Dict[str, float]:
    was_training = model.training
    model.eval()
    try:
        prompt = build_prompt(task_id)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                           max_length=MAX_PROMPT_TOK).to(model.device)
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                  # greedy
            pad_token_id=tokenizer.eos_token_id,
        )
        completion = tokenizer.decode(
            output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
        )
    finally:
        if was_training:
            model.train()
    plan = parse_plan(completion)
    rec = rollout_score(task_id, plan)
    rec.update({"plan": plan, "completion": completion[:600]})
    return rec


def evaluate(label: str) -> Dict[str, Dict]:
    results = {}
    print(f"\n=== {label} evaluation ({len(TASK_IDS)} tasks) ===")
    for tid in TASK_IDS:
        rec = policy_rollout(tid)
        results[tid] = rec
        print(f"  {tid:<8s}  reward={rec['reward']:.3f}  steps={rec['steps']:>2d}  "
              f"plan_len={len(rec['plan'])}")
    avg = mean(r["reward"] for r in results.values())
    print(f"--- {label} mean grader score: {avg:.3f}")
    return results

## 6.5 · Baseline evaluation (untrained model)

In [ ]:
baseline_results = evaluate("BASELINE (untrained)")

## 7 · Phase A — SFT warm-start on expert plans

We run a short, prompt-masked SFT pass on the 15 expert plans shipped with the
environment (`agent.TASKS_COMMANDS`). The loss is computed only over the
command tokens (the prompt is masked with `-100`), so the model learns the
**output channel** — one verb per line, ending with `FINALIZE_SPRINT` —
without ever seeing the grader.

This step is what prevents pure-GRPO format collapse on a 1.5 B model:
after SFT the policy already produces parseable plans, so GRPO has a useful
reward gradient to push on.

In [ ]:
def build_sft_pairs():
    pairs = []
    for tid in TASK_IDS:
        if tid not in TASKS_COMMANDS:
            continue
        prompt = build_prompt(tid)
        target = "\n".join(TASKS_COMMANDS[tid])
        pairs.append((prompt, target))
    return pairs


def encode_pair(prompt: str, target: str):
    """Tokenize prompt+target, mask prompt tokens with -100 in labels."""
    prompt_full = prompt + "\n"
    full_text = prompt_full + target + tokenizer.eos_token
    full_ids = tokenizer(full_text, return_tensors="pt", add_special_tokens=False).input_ids[0]
    prompt_ids = tokenizer(prompt_full, return_tensors="pt", add_special_tokens=False).input_ids[0]
    labels = full_ids.clone()
    labels[: len(prompt_ids)] = -100
    return full_ids, labels


sft_pairs = build_sft_pairs()
print(f"SFT pairs: {len(sft_pairs)}")

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=SFT_LR)

model.train()
global_step = 0
t0 = time.time()
for epoch in range(SFT_EPOCHS):
    random.shuffle(sft_pairs)
    losses = []
    for prompt, target in sft_pairs:
        ids, labels = encode_pair(prompt, target)
        ids = ids.unsqueeze(0).to(model.device)
        labels = labels.unsqueeze(0).to(model.device)
        out = model(input_ids=ids, labels=labels)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, SFT_GRAD_CLIP)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        losses.append(loss.item())
        global_step += 1
    print(f"SFT epoch {epoch+1}/{SFT_EPOCHS}  steps={global_step}  mean_loss={mean(losses):.4f}")
print(f"SFT done in {time.time() - t0:.1f}s")
model.eval()

## 7.5 · SFT evaluation

In [ ]:
sft_results = evaluate("AFTER SFT")

## 8 · Phase B — GRPO refinement

Now that the policy already speaks SprintBoard, GRPO can sample multiple
plans per scenario, score them with the trajectory reward, and push the
policy toward the higher-reward variants.

We use a small LR and `beta=0` so the SFT-warmed adapter is preserved (with
PEFT, the GRPO reference policy is the *base* model, so a non-zero KL term
would actively undo SFT).

In [ ]:
training_reward_log.clear()
training_format_log.clear()


class SprintBoardProgressCallback(TrainerCallback):
    """Print one line every logging step (loss, etc.) + last mean env reward."""

    def __init__(self, reward_log: list, format_log: list):
        self.reward_log = reward_log
        self.format_log = format_log

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not state.is_world_process_zero or not logs:
            return
        parts = [f"step {state.global_step}"]
        for key in ("loss", "grad_norm", "learning_rate"):
            if key in logs and isinstance(logs[key], (int, float)):
                parts.append(f"{key}={float(logs[key]):.4g}")
        if self.reward_log:
            parts.append(f"env_rwd={self.reward_log[-1]:.3f}")
        if self.format_log:
            parts.append(f"fmt_rwd={self.format_log[-1]:.3f}")
        print(" | ".join(parts), flush=True)


config = GRPOConfig(
    output_dir=str(OUT_DIR),
    learning_rate=GRPO_LR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_NEW_TOKENS,
    max_prompt_length=MAX_PROMPT_TOK,
    num_train_epochs=GRPO_EPOCHS,
    logging_steps=1,
    logging_first_step=True,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
    save_strategy="no",
    dataloader_num_workers=0,
    remove_unused_columns=False,
    beta=GRPO_BETA,
    seed=42,
    disable_tqdm=False,
    temperature=0.7,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[trajectory_reward, format_reward],
    args=config,
    train_dataset=dataset,
    callbacks=[SprintBoardProgressCallback(training_reward_log, training_format_log)],
)

t0 = time.time()
train_result = trainer.train()
print(f"\nGRPO finished in {time.time() - t0:.1f}s")

## 8.5 · Trained-policy evaluation

In [ ]:
trained_results = evaluate("TRAINED (post-GRPO)")

## 9 · Plots & summary saved to `assets/`

Three PNGs are saved so the README can embed them:

* `assets/grpo_reward_curve.png` — mean trajectory reward + format reward per GRPO step.
* `assets/before_after_per_task.png` — baseline / SFT / trained per task.
* `assets/training_summary.json` — numeric summary.

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 4.4))
if training_reward_log:
    xs = range(1, len(training_reward_log) + 1)
    ax.plot(xs, training_reward_log, marker="o", linewidth=2,
            color="#7c3aed", label="Trajectory reward")
    if len(training_reward_log) >= 5:
        rolling = np.convolve(training_reward_log, np.ones(5)/5, mode="valid")
        ax.plot(range(5, len(training_reward_log) + 1), rolling,
                color="#22c55e", linewidth=2, label="5-step rolling mean")
if training_format_log:
    xf = range(1, len(training_format_log) + 1)
    ax.plot(xf, training_format_log, marker="x", linewidth=1, alpha=0.7,
            color="#f59e0b", label="Format reward")
ax.set_xlabel("GRPO update step")
ax.set_ylabel("Reward")
ax.set_ylim(0, 1.0)
ax.set_title("SprintBoard GRPO training rewards")
ax.grid(alpha=0.3, linestyle="--")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(ASSETS_DIR / "grpo_reward_curve.png", dpi=180)
plt.show()

task_ids = list(baseline_results.keys())
base_vals = [baseline_results[t]["reward"] for t in task_ids]
sft_vals = [sft_results[t]["reward"] for t in task_ids]
train_vals = [trained_results[t]["reward"] for t in task_ids]
delta_sft = mean(sft_vals) - mean(base_vals)
delta_grpo = mean(train_vals) - mean(base_vals)

fig, ax = plt.subplots(figsize=(11.5, 4.8))
x = np.arange(len(task_ids))
w = 0.27
ax.bar(x - w, base_vals, w, label=f"Baseline ({mean(base_vals):.2f})",
       color="#94a3b8", edgecolor="#475569")
ax.bar(x, sft_vals, w, label=f"After SFT ({mean(sft_vals):.2f})",
       color="#0ea5e9", edgecolor="#0369a1")
ax.bar(x + w, train_vals, w, label=f"After GRPO ({mean(train_vals):.2f})",
       color="#7c3aed", edgecolor="#5b21b6")
ax.set_xticks(x); ax.set_xticklabels([t.replace("task_", "T") for t in task_ids])
ax.set_ylim(0, 1.05)
ax.set_xlabel("Task"); ax.set_ylabel("Grader score (0-1)")
ax.set_title(f"SprintBoard — baseline vs SFT vs GRPO  "
             f"(delta_SFT={delta_sft:+.2f}, delta_GRPO={delta_grpo:+.2f})")
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(ASSETS_DIR / "before_after_per_task.png", dpi=180)
plt.show()

summary = {
    "model": MODEL_ID,
    "sft_epochs": SFT_EPOCHS,
    "grpo_epochs": GRPO_EPOCHS,
    "baseline_mean": round(mean(base_vals), 4),
    "sft_mean":      round(mean(sft_vals), 4),
    "trained_mean":  round(mean(train_vals), 4),
    "delta_sft":     round(delta_sft, 4),
    "delta_grpo":    round(delta_grpo, 4),
    "per_task": {
        t: {"baseline": round(b, 4), "sft": round(s, 4), "trained": round(tr, 4)}
        for t, b, s, tr in zip(task_ids, base_vals, sft_vals, train_vals)
    },
}
(ASSETS_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("\nPlots saved ->", ASSETS_DIR.resolve())

## 10 · (Optional) Push the LoRA adapter to the Hub

In [ ]:
PUSH = False  # flip to True after `huggingface-cli login`
REPO_ID = "your-username/sprintboard-qwen25-1.5b-grpo"

if PUSH:
    model.push_to_hub(REPO_ID, private=False)
    tokenizer.push_to_hub(REPO_ID)
    print("Pushed to", REPO_ID)
else:
    print("Set PUSH = True and authenticate to publish the adapter.")